In [ ]:
# Import Modules
!pip install argon2-cffi
import argon2, time, json, os, hmac, hashlib, struct, secrets, ctypes

In [ ]:
# Add Constants
MAX_FAILED_LOGINS = 5
TIMEOUT_MILLIS = 900000
PASSWORD_MIN_LENGTH = 16
USER_DATA_FILE = "user_data.json"
OTP_EXPIRY_SECONDS = 60

# Define Functions
"""
Returns the current time in milliseconds
"""
def get_time() -> int:
    return int((time.time() * 1000))

"""
Checks to make sure the password meets security requirements

username: str - Account username. Used to verify it is not present in password
password: str - Account password.
"""
def check_password_quality(username: str, password: str) -> bool:
    if (len(password) < PASSWORD_MIN_LENGTH):
        print(f"Password shorter than minimum L length ({PASSWORD_MIN_LENGTH})")
        return None
    if (username.lower() in password.lower()):
        print("Username cannot be present in password")
        return None
    return True

"""
Common function for all login failures. Always increments failed login count. Ensures auth failures are uniform and different errors will not leak information

username: str - Account to increment failed login count
"""
def fail_login(username: str) -> bool:
    data: dict = load_user_data()
    failed_logins: int

    if (username not in data.keys()):
        print(f"Error: Username not found in database")
        return False
    
    print("Login Failure")
    failed_logins = data[username]['failed_logins']
    data[username]['failed_logins'] = failed_logins + 1

    save_user_data(data)
    return True

"""
Checks if an account is locked out.
Sets lockout parameters as appropriate. If timeout has elapsed, reset all data.

timeout_reset_millis: int - Last time, in milliseconds, the timeout counter was reset
failed_logins: int - Number of failed login attempts within lockout period. Should be reset once timeout elapses
"""
def check_lockout(username: str, lockout_time_mills: int, failed_logins: int) -> bool:
    current_time_millis: int = get_time()
    timeout_delta: int =  current_time_millis - lockout_time_mills
    data: dict = load_user_data()

    if (username not in data.keys()):
        print(f"Error: Username not found in database")
        return False

    data[username]['lockout_time'] = get_time()
    if (timeout_delta > TIMEOUT_MILLIS):
        data[username]['failed_logins'] = 0
        return False
    elif (failed_logins < MAX_FAILED_LOGINS):
        return False
    
    save_user_data(data)
    return True

"""
Verifies password against existing hash

ph: PasswordHasher - an argon2 PasswordHasher for verification
provided_password: str - Password input by the user
hash: str - hash stored in user account data
"""
def verify_password(ph: argon2.PasswordHasher, provided_password: str, hash: str) -> bool:
    try:
        return ph.verify(hash, provided_password)
    except argon2.exceptions.VerifyMismatchError:
        return False

"""
Initializes PasswordHasher
"""
def init_password_hasher() -> argon2.PasswordHasher:
    return argon2.PasswordHasher(time_cost=3, memory_cost=65536, salt_len=256, hash_len=32, parallelism=8)

"""
Loads user data from the JSON file.
Returns an empty dict if the file does not exist.
"""
def load_user_data() -> dict:
    if not os.path.exists(USER_DATA_FILE):
        return {}
    try:
        with open(USER_DATA_FILE, "r") as f:
            return json.load(f)
    except (json.JSONDecodeError, IOError) as e:
        print(f"Warning: Could not read user data: {e}")
        return {}

"""
Saves user data to the JSON file.
Uses an atomic write to prevent data corruption on crash.

data: dict - Full user database to persist
"""
def save_user_data(data: dict) -> None:
    try:
        temp_path = USER_DATA_FILE + ".tmp"
        with open(temp_path, "w") as f:
            json.dump(data, f, indent=4)
        os.replace(temp_path, USER_DATA_FILE)
    except IOError as e:
        print(f"Error: Failed to save user data: {e}")

"""
Creates a new user record with all L fields initialized.
Tracks: password hash, TOTP secret, failed attempts, lockout time,
OTP, OTP expiration, and reset token.

username: str - Username for the new account
"""
def initialize_user_record(username: str) -> dict:
    return {
        "username":                 username,
        "password_hash":            None,
        "totp_secret":              secrets.token_hex(20),
        "failed_attempts":          0,
        "lockout_time":             get_time(),
        "otp":                      None,
        "otp_expiration":           None,
    }

"""
Best-effort secure erasure of a plaintext string from memory.
Uses ctypes.memset to zero the object's memory address, reducing the
window in which the plaintext could be extracted from a memory dump.

plaintext: str - The sensitive string to zero out
"""
def secure_erase(plaintext: str) -> None:
    if plaintext is None:
        return
    try:
        ctypes.memset(id(plaintext), 0, len(plaintext.encode("utf-8")))
    except Exception:
        pass

"""
Registers a new user account.
Validates password quality, hashes the password with Argon2,
creates a user record, and persists it to the JSON file.

username: str - Desired account username
password: str - Plaintext password (erased from memory after hashing)
"""
def register_user(username: str, password: str) -> bool:
    data = load_user_data()
    if username in data:
        print("Username already exists")
        return False
    if not check_password_quality(username, password):
        return False
    ph     = init_password_hasher()
    hash   = ph.hash(password)
    secure_erase(password)
    record = initialize_user_record(username)
    record["password_hash"] = hash
    data[username] = record
    save_user_data(data)
    print(f"User '{username}' registered successfully")
    return True

"""
Authenticates a user with their password.
Checks lockout status, verifies credentials, and resets failed attempts on success.
Calls fail_login() on any failure to ensure uniform error handling.

username: str - Account username
password: str - Plaintext password (erased from memory after verification)
"""
def login_user(username: str, password: str) -> bool:
    data = load_user_data()
    if username not in data:
        fail_login(username)
        return False
    user = data[username]
    if check_lockout(username, user["lockout_time"], user["failed_attempts"]):
        print("Account is locked. Please try again later.")
        return False
    ph = init_password_hasher()
    if not verify_password(ph, password, user["password_hash"]):
        secure_erase(password)
        fail_login(username)
        return False
    secure_erase(password)
    data[username]["failed_attempts"] = 0
    data[username]["lockout_time"]    = get_time()
    save_user_data(data)
    return True

"""
Generates a 6-digit Time-Based One-Time Password using HMAC-SHA1.
Follows RFC 6238: computes a 30-second time step, packs it as an
8-byte big-endian counter, applies HMAC-SHA1, then dynamically
truncates the digest to produce a 6-digit code.

secret_key: str - Hex-encoded TOTP secret stored in the user record
"""
def generate_totp(secret_key: str) -> str:
    time_step  = int(time.time()) // 30
    time_bytes = struct.pack(">Q", time_step)
    key_bytes  = bytes.fromhex(secret_key)
    digest     = hmac.new(key_bytes, time_bytes, hashlib.sha1).digest()
    offset     = digest[-1] & 0x0F
    raw_code   = struct.unpack(">I", digest[offset:offset + 4])[0] & 0x7FFFFFFF
    return f"{raw_code % 1_000_000:06d}"

"""
Simulates an authenticator app by generating a TOTP for the user,
storing it with a 60-second expiration, and printing it to the console.

username: str - Account requesting a TOTP
"""
def simulate_authenticator(username: str) -> str:
    data = load_user_data()
    if username not in data:
        print(f"User '{username}' not found")
        return None
    totp = generate_totp(data[username]["totp_secret"])
    data[username]["otp"]            = totp
    data[username]["otp_expiration"] = time.time() + OTP_EXPIRY_SECONDS
    save_user_data(data)
    print(f"\n[Authenticator App] Your TOTP: {totp}")
    print(f"[Authenticator App] This code expires in {OTP_EXPIRY_SECONDS} seconds.\n")
    return totp

"""
Validates the TOTP entered by the user to complete the 2FA login step.
Rejects expired or missing OTPs. Uses hmac.compare_digest for constant-time
comparison to prevent timing attacks. Invalidates the OTP after one successful
use to prevent replay attacks.

username    : str - Account completing 2FA
entered_totp: str - 6-digit code entered by the user
"""
def verify_totp(username: str, entered_totp: str) -> bool:
    data = load_user_data()
    if username not in data:
        print(f"User '{username}' not found")
        return False
    user           = data[username]
    stored_otp     = user.get("otp")
    otp_expiration = user.get("otp_expiration", 0)
    if stored_otp is None:
        print("No active OTP found. Please request a new code.")
        return False
    if time.time() > otp_expiration:
        print("OTP has expired. Please request a new code.")
        data[username]["otp"]            = None
        data[username]["otp_expiration"] = None
        save_user_data(data)
        return False
    if hmac.compare_digest(str(stored_otp), str(entered_totp)):
        print("TOTP verified. Login successful!")
        data[username]["otp"]            = None
        data[username]["otp_expiration"] = None
        save_user_data(data)
        return True
    print("Invalid TOTP. Access denied.")
    return False

def main():
    while True:
        print("\n(1) Register")
        print("(2) Login")
        print("(3) View User Data")
        print("(4) Exit")
        choice = input("Select an option: ").strip()

        if (choice == "1"):
            username = input("Enter username: ").strip()
            password = input("Enter password: ").strip()
            register_user(username, password)

        elif (choice == "2"):
            username = input("Enter username: ").strip()
            password = input("Enter password: ").strip()
            if login_user(username, password):
                simulate_authenticator(username)
                entered_totp = input("Enter TOTP: ").strip()
                verify_totp(username, entered_totp)

        elif (choice == "3"):
            print(json.dumps(load_user_data(), indent=4))

        elif (choice == "4"):
            print("Exiting...")
            break

        else:
            print("Invalid option. Please try again.")

In [ ]:
if __name__ == "__main__":
        main()